# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a guided workflow for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)


In [ ]:
# Ensure `mlcroissant` and pandas are installed
!pip install mlcroissant pandas

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant metadata schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets and their fields by their @id

record_sets = metadata.record_sets
if not record_sets:
    print("No record sets defined in the schema.")
else:
    for rs in record_sets:
        print(f"RecordSet name: {rs.name}")
        print(f"  @id: {rs.id}")
        print("  Fields:")
        if hasattr(rs, 'fields') and rs.fields:
            for field in rs.fields:
                col_ids = [col.id for col in field.columns] if hasattr(field, 'columns') and field.columns else []
                print(f"    - {field.name} (field @id: {field.id}, column @id(s): {col_ids})")
        else:
            print("    (No fields listed)")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Prepare to extract data from each record set
# 
# Select the main record set by its @id. We expect record_sets to contain a primary record set, e.g. '@id': 'https://api.app.sen.science/frontiers/7160411/recs1'

# Get all record set @ids:
record_set_ids = [rs.id for rs in metadata.record_sets]
print("Record sets @id:", record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nLoading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")
    else:
        print("No records found for record set.")

# Choose the main record set for further analysis (replace the @id if more than one):
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id is not None:
    print(f"\nSample data from record set '{main_record_set_id}':")
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets available for extraction.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming distributions, or grouping data by key attributes.


In [ ]:
# Conduct EDA on the main record set
import numpy as np

if main_record_set_id is not None:
    df = dataframes[main_record_set_id].copy()

    # Print all column names for EDA reference
    print("Columns present:", df.columns.tolist())

    # Try to identify a numeric field (commonly: GAD-7, PHQ-9, PSQ score)
    candidate_score_fields = [
        c for c in df.columns if any(substr in c.lower() for substr in ["gad", "phq", "psq", "score", "age"])
    ]
    print("Numeric field candidates:", candidate_score_fields)

    # Use the first available candidate as the numeric field, or ask user to adjust
    if candidate_score_fields:
        numeric_field = candidate_score_fields[0]  # Or choose as appropriate
        print(f"\nUsing '{numeric_field}' as numeric field for filtering and normalization.")

        # Remove missing or non-numeric values if present
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        df = df.dropna(subset=[numeric_field])

        # Example threshold (can be adjusted based on field)
        threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 0
        print(f"Threshold for filtering: {threshold}")
        filtered_df = df[df[numeric_field] > threshold]

        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by a categorical field, e.g. 'gender', 'age_group', 'education'
        group_field_candidates = [c for c in df.columns if any(key in c.lower() for key in ["gender", "sex", "education", "age_group"])]
        print("Possible grouping fields:", group_field_candidates)

        if group_field_candidates:
            group_field = group_field_candidates[0]
            print(f"\nGrouping by '{group_field}':")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No candidate numeric field found for analysis.")
else:
    print("EDA skipped: No main record set DataFrame.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and candidate_score_fields:
    # Plot distribution of the main score
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If grouping field, show boxplot
    if group_field_candidates:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization skipped: No suitable numeric data found.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded and explored a mental health survey dataset from Kilifi County, Kenya using the `mlcroissant` library.
- Data includes demographic, GAD-7, PHQ-9, and PSQ variables; structure and field names discovered using metadata.
- Performed initial EDA and visualizations on key numeric fields grouped by demographic categories, paving the way for further detailed analysis.
